In [ ]:
import itertools
from enum import Enum

import numpy as np
import pandas as pd
import scipy.stats as stats


In [2]:
# df = pd.read_csv("../../../cluster_results/pegasus/final_final_versions_d54c3e0a/uc1_ur10_collision.csv")
df = pd.read_csv("../../../cluster_results/pegasus/final_final_versions_d54c3e0a/complex_env_dual_arm_collision.csv")
# df = pd.read_csv("../../../cluster_results/pegasus/final_final_versions_d54c3e0a/complex_env_dual_arm_collision_max256verts.csv")
df.set_index("file", inplace=True)
rng = np.random.default_rng(42)

from algo_metadata import AlgoData, algo_metadata


In [3]:
EXCLUDED_ALGORITHMS = [
    "openGJK distance linear support",
    "libccd intersection linear support",
    "FCL intersection linear support",
    "FCL distance linear support",
]

N_RESAMPLES = 9999
CONFIDENCE_LEVEL = 0.95
RANDOM_SEED = 42


In [ ]:
def geometric_mean_ratio(fast, slow):
    """Returns the geometric mean of the ratio slow/fast.

    """
    fast = np.asarray(fast, dtype=float)
    slow = np.asarray(slow, dtype=float)
    return np.exp(np.mean(np.log(slow / fast), axis=-1))


def speedup_with_ci(fast_values, slow_values, rng):
    """Calculate the speedup factor of "fast_values" over "slow_values" with a 95% CI.
    """
    paired = pd.DataFrame({"fast": fast_values, "slow": slow_values}).dropna()

    fast = paired["fast"].to_numpy()
    slow = paired["slow"].to_numpy()

    point = geometric_mean_ratio(fast, slow)
    result = stats.bootstrap(
        (fast, slow),
        statistic=geometric_mean_ratio,
        paired=True,
        vectorized=True,
        n_resamples=N_RESAMPLES,
        confidence_level=CONFIDENCE_LEVEL,
        method="BCa",
        rng=rng,
    )
    return {
        "point": float(point),
        "ci_low": float(result.confidence_interval.low),
        "ci_high": float(result.confidence_interval.high),
        "n": len(paired),
    }


In [ ]:
def speedup_rows_to_latex(rows, caption, label):
    """rows: list of dicts with keys faster, slower, point, ci_low, ci_high, n."""
    def esc(s):
        return s.replace("_", "\\_")

    lines = [
        "\\begin{table}[htbp]",
        "\\centering",
        "\\begin{tabular}{llccc}",
        "\\toprule",
        "Faster & Slower & Speedup & 95\\% CI & $n$ \\\\",
        "\\midrule",
    ]
    for r in rows:
        n_str = str(r["n"]) if not isinstance(r["n"], tuple) else f"{r['n'][0]}/{r['n'][1]}"
        lines.append(
            f"{esc(r['faster'])} & {esc(r['slower'])} & "
            f"{r['point']:.2f}$\\times$ & [{r['ci_low']:.2f}, {r['ci_high']:.2f}] & {n_str} \\\\"
        )
    lines += [
        "\\bottomrule",
        "\\end{tabular}",
        f"\\caption{{{caption}}}",
        f"\\label{{{label}}}",
        "\\end{table}",
    ]
    return "\n".join(lines)


In [6]:
# analysis

def fastest_algorithm_per_language(df, category: AlgoData.Category):
    """Return {language: raw_algorithm_name} for the given category, based on median runtime."""
    best = {}
    for algo, meta in algo_metadata.items():
        if algo in EXCLUDED_ALGORITHMS or algo not in df.columns:
            continue
        if meta.category is not category:
            continue
        median = df[algo].median(skipna=True)
        lang = meta.language.value
        if lang not in best or median < best[lang][1]:
            best[lang] = (algo, median)
    return {lang: algo for lang, (algo, _) in best.items()}



for category in AlgoData.Category:
    best_per_language = fastest_algorithm_per_language(df, category)
    if len(best_per_language) < 2:
        print(f"--- {category.value} ---")
        print("Fewer than two languages have a remaining algorithm -- skipping.\n")
        continue

    print(f"--- {category.value}: fastest algorithm per language ---")
    for lang, algo in sorted(best_per_language.items(), key=lambda kv: df[kv[1]].median()):
        pretty = algo_metadata[algo].pretty_name
        print(f"  {lang}: {pretty} (median {df[algo].median():.2f} µs)")
    print()

    ordered_langs = sorted(best_per_language, key=lambda lang: df[best_per_language[lang]].median())

    rows = []
    for lang_fast, lang_slow in itertools.combinations(ordered_langs, 2):
        algo_fast = best_per_language[lang_fast]
        algo_slow = best_per_language[lang_slow]

        result = speedup_with_ci(df[algo_fast], df[algo_slow], rng)

        pretty_fast = algo_metadata[algo_fast].pretty_name
        pretty_slow = algo_metadata[algo_slow].pretty_name

        print(
            f"{pretty_fast} ({lang_fast}) is on average {result['point']:.2f}x faster than "
            f"{pretty_slow} ({lang_slow}) "
            f"[95% CI: {result['ci_low']:.2f}x - {result['ci_high']:.2f}x, "
            f"n={result['n']}]"
        )

        rows.append({
            "faster": f"{pretty_fast} ({lang_fast})",
            "slower": f"{pretty_slow} ({lang_slow})",
            "point": result["point"],
            "ci_low": result["ci_low"],
            "ci_high": result["ci_high"],
            "n": result["n"],
        })

    print()
    print(speedup_rows_to_latex(
        rows,
        caption=f"Speedup factor (95% bootstrap CI) between the fastest {category.value.lower()} "
                f"algorithm of each language.",
        label=f"tab:speedup-best-per-language-{category.value.lower()}",
    ))
    print("\n")



--- Intersection: fastest algorithm per language ---
  C++: libccd (median 1.51 µs)
  Rust: collision-rs (median 28.77 µs)
  Python: distance3d Jolt (median 131.55 µs)

libccd (C++) is on average 23.02x faster than collision-rs (Rust) [95% CI: 21.81x - 24.39x, n=1000]
libccd (C++) is on average 107.96x faster than distance3d Jolt (Python) [95% CI: 102.09x - 114.49x, n=1000]
collision-rs (Rust) is on average 4.69x faster than distance3d Jolt (Python) [95% CI: 4.53x - 4.85x, n=1000]

\begin{table}[htbp]
\centering
\begin{tabular}{llccc}
\toprule
Faster & Slower & Speedup & 95\% CI & $n$ \\
\midrule
libccd (C++) & collision-rs (Rust) & 23.02$\times$ & [21.81, 24.39] & 1000 \\
libccd (C++) & distance3d Jolt (Python) & 107.96$\times$ & [102.09, 114.49] & 1000 \\
collision-rs (Rust) & distance3d Jolt (Python) & 4.69$\times$ & [4.53, 4.85] & 1000 \\
\bottomrule
\end{tabular}
\caption{Speedup factor (95% bootstrap CI) between the fastest intersection algorithm of each language.}
\label{tab:spe